# Information Health — one-click Colab demo

Runs the **FastAPI engine** + a **production build** of the Next.js app inside Colab and
opens a public URL.

- A *production* build is served on purpose: `next dev` injects CSS via JS and uses a
  hot-reload websocket that misbehaves behind a tunnel (blank / unstyled page). Production
  ships a real stylesheet + static chunks, so it renders reliably.
- The onboarding → **Initial Information Health Estimate** flow needs no credentials.
- **See the whole signed-in app with no Google setup:** a one-click *“Continue as demo reader”*
  login (dev-only) plus a cell that pre-loads sample reads means the **Dashboard, Reading History,
  Analytics, and Profile** are populated the moment you sign in.
- Optional cells: **real publisher names** (Qbias profile) and **Google sign-in**.

Run the cells top to bottom (Runtime → *Run all* also works).

> If your repo is **private**, paste a GitHub token (read access) in step 1.


In [ ]:
#@title 1 · Clone the repo + install Node 20
REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}   # only needed if the repo is private

auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
!rm -rf app && git clone --depth 1 --branch {BRANCH} https://{auth}github.com/{REPO}.git app
%cd app
# Colab ships an old Node; install a modern one for Next.js 14.
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1
!sudo apt-get install -y nodejs >/dev/null 2>&1
!node -v ; npm -v

In [ ]:
#@title 2 · Start the FastAPI engine (synthetic corpus, no external data)
import subprocess, time, urllib.request

!pip install -q -e ".[serve]"

def wait(url, n=180):
    for _ in range(n):
        try:
            if urllib.request.urlopen(url, timeout=2).status == 200: return True
        except Exception: time.sleep(1)
    return False

engine = subprocess.Popen(["python", "examples/api_fastapi.py"],
                          stdout=open("engine.log", "w"), stderr=subprocess.STDOUT)
print("engine:", "UP" if wait("http://127.0.0.1:8000/api/health") else "FAILED — see engine.log")

In [ ]:
#@title 3 · Build the web app (production) + start it, pointed at the engine
import secrets
# Production is served (dev mode's HMR + JS-injected CSS break behind a tunnel).
# Prod also disables the mock fallback, so it uses the real engine from step 2.
# RWE_DEV_LOGIN / NEXT_PUBLIC_DEV_LOGIN turn on a one-click "Continue as demo reader" sign-in so you
# can explore the full signed-in app (Dashboard, History, Analytics, Profile, Settings) without
# Google OAuth. Dev/demo ONLY — never set these in a real deployment.
open("web/.env.local", "w").write(
    "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
    f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
    "RWE_DEV_LOGIN=1\n"
    "NEXT_PUBLIC_DEV_LOGIN=1\n")
!cd web && npm install --no-audit --no-fund --loglevel=error
!cd web && npm run build
web = subprocess.Popen(["npm", "start"], cwd="web",
                       stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
print("web:", "UP" if wait("http://127.0.0.1:3000/onboarding") else "still starting — check web.log")

In [ ]:
#@title 4 · Open it — public URL via a Cloudflare quick tunnel  (safe to re-run)
import re, time, subprocess, secrets, os

# Clean any previous tunnel so re-running this cell is idempotent.
subprocess.run(["pkill", "-f", "cloudflared"], stderr=subprocess.DEVNULL); time.sleep(1)
# (Re)download cloudflared if it's missing or truncated.
if not os.path.exists("cloudflared") or os.path.getsize("cloudflared") < 1_000_000:
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   "cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)
print("cloudflared binary:", os.path.getsize("cloudflared"), "bytes")

open("cf.log", "w").close()
cf = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:3000", "--no-autoupdate"],
                      stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(90):                       # quick tunnels can take 20-40s — wait up to ~90s
    time.sleep(1)
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", open("cf.log").read())
    if m: PUBLIC_URL = m.group(0); break

if not PUBLIC_URL:
    print("\n⚠️  No tunnel URL yet — Cloudflare quick tunnels are occasionally slow/rate-limited.")
    print("    Just RE-RUN THIS CELL (it usually works on the 2nd try). Recent cloudflared log:")
    print("    " + "\n    ".join(open("cf.log").read().splitlines()[-12:] or ["(empty)"]))
else:
    # Point NextAuth at the public tunnel URL and restart the web server, so sign-in redirects target
    # the tunnel (not the container's localhost). Runtime env only — no rebuild, so the demo-login
    # button baked in step 3 stays; the dev demo login stays enabled.
    open("web/.env.local", "w").write(
        "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
        f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
        f"NEXTAUTH_URL={PUBLIC_URL}\n"
        "RWE_DEV_LOGIN=1\n"
        "NEXT_PUBLIC_DEV_LOGIN=1\n")
    subprocess.run(["pkill", "-f", "next-server"], stderr=subprocess.DEVNULL); time.sleep(2)
    web = subprocess.Popen(["npm", "start"], cwd="web",
                           stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
    wait("http://127.0.0.1:3000/onboarding")
    print("\n👉  Open:", PUBLIC_URL)
    print("    Anonymous : /onboarding → pick publishers → Initial Estimate → Report / Recommendations / Coach")
    print("    Whole app : Sign in → “Continue as demo reader” (the only button) → Dashboard / History / Analytics / Profile")
    print("    (run step 5 first to pre-load the demo reader with sample reads)")

In [ ]:
#@title 5 · Pre-load the demo reader with sample reads (so the signed-in pages have data)
# Creates the SAME throwaway demo account the "Continue as demo reader" button signs into, and adds a
# handful of real-outlet reads across the spectrum — enough to cross the measured-report threshold —
# so Dashboard / History / Analytics / Report populate the moment you sign in. Dev/demo only; the
# engine trusts the local web tier here because no RWE_INTERNAL_SECRET is set in this demo.
import json, urllib.request
ENGINE = "http://127.0.0.1:8000"
def _post(path, body, headers=None):
    req = urllib.request.Request(ENGINE + path, data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json", **(headers or {})})
    with urllib.request.urlopen(req, timeout=15) as r:
        return json.loads(r.read().decode())

uid = _post("/api/internal/users",
            {"provider": "dev", "providerAccountId": "demo@infodiet.local",
             "email": "demo@infodiet.local", "displayName": "Demo Reader"})["userId"]
SAMPLE = [
    ("https://www.nytimes.com/2026/us/politics/senate-vote", "Senate advances the funding bill, leaders say"),
    ("https://www.foxnews.com/politics/border-plan", "Outrage as officials slam the border plan"),
    ("https://www.wsj.com/economy/inflation-opinion", "Opinion: why we must rethink inflation policy"),
    ("https://www.washingtonpost.com/politics/court-analysis", "Analysis: what to know about the court ruling"),
    ("https://www.theguardian.com/us-news/climate-deal", "Hope as historic climate deal is celebrated"),
    ("https://apnews.com/hub/politics/poll", "Poll finds shifting views on the economy, new data shows"),
    ("https://www.npr.org/2026/health/study", "Study finds a new treatment shows promise, researchers say"),
    ("https://www.cnn.com/2026/tech/ai-regulation", "Lawmakers clash over AI regulation amid fierce debate"),
]
res = _post("/api/me/reads", {"reads": [{"url": u, "title": t} for u, t in SAMPLE]},
            {"X-IH-User-Id": str(uid)})
print(f"demo reader #{uid}: {res['totalReads']} reads stored — measured report ready: {res['sufficient']}")
print("→ open the URL, Sign in → “Continue as demo reader”, and your Dashboard/History/Analytics are populated.")

## Optional — real publisher names (Qbias)

The default `synthetic` profile shows generated outlet names ("Outlet N") with real
leans. To show **real publishers** (NYT, Fox, WSJ, NPR…) with real AllSides gold lean,
switch the engine to the **Qbias** profile below — configuration only, no app change.
Run it any time after step 3, then reload the app.


In [ ]:
#@title (optional) · Real publisher names — switch the engine to the Qbias AllSides catalog
import subprocess, time, os
# Config only: real outlets + real AllSides gold lean (~14 MB download).
!wget -q "https://raw.githubusercontent.com/irgroup/Qbias/main/allsides_balanced_news_headlines-texts.csv" -O qbias.csv
print("csv size:", os.path.getsize("qbias.csv"), "bytes")   # ~13.7M expected; ~0 means the download failed
try: engine.terminate(); time.sleep(2)
except Exception: pass
env = {**os.environ, "RWE_PROFILE": "qbias", "RWE_QBIAS": "qbias.csv",
       "RWE_MAX_ITEMS": "1200", "RWE_N_USERS": "300"}
engine = subprocess.Popen(["python", "examples/api_fastapi.py"], env=env,
                          stdout=open("engine.log", "w"), stderr=subprocess.STDOUT)
print("engine (qbias):", "UP" if wait("http://127.0.0.1:8000/api/health", 240) else "check engine.log")
print("→ reload the app — the onboarding outlet chips now show real publishers")

## Optional — live RSS feed (real publishers **with openable article URLs**)

The Qbias cell above shows real outlet names, but that static catalog has **no article URLs** — a
recommendation's **Read** button records the open yet has nowhere to navigate.

This cell ingests real, cross-spectrum **RSS feeds** into the news catalog and switches the engine
to draw recommendations from it (`RWE_RECS_SOURCE=feed`). Each recommendation then carries the
**real publisher URL**, so **Read** opens the actual article — and the browser extension can capture
that read, so Dashboard / History / Analytics / Open-Mindedness update naturally. Needs outbound
network; if the feeds are unreachable it falls back to the previous corpus (no URLs), so it is
always safe to run. Edit `deploy/rss_feeds.example.txt` to change the feed mix.

In [ ]:
#@title (optional) · Live RSS feed — recommendations that carry real, openable publisher URLs
import subprocess, time, os, json, urllib.request

# 1) Ingest cross-spectrum RSS/Atom feeds into the FeedArticle catalog (edit the feeds file to taste).
#    Tolerates per-feed failures; the engine needs >= RWE_FEED_MIN_ARTICLES (default 50) to switch.
!python examples/rss_ingest.py run --feeds deploy/rss_feeds.example.txt
!python examples/rss_ingest.py status

# 2) Restart the engine SOURCING recommendations from the live catalog. With a big-enough catalog it
#    builds the corpus from the feed (real names + real URLs); otherwise it falls back safely.
try: engine.terminate(); time.sleep(2)
except Exception: pass
env = {**os.environ, "RWE_RECS_SOURCE": "feed", "RWE_MAX_ITEMS": "1200", "RWE_N_USERS": "300"}
engine = subprocess.Popen(["python", "examples/api_fastapi.py"], env=env,
                          stdout=open("engine.log", "w"), stderr=subprocess.STDOUT)
print("engine (live feed):", "UP" if wait("http://127.0.0.1:8000/api/health", 240) else "check engine.log")

# 3) Confirm recommendations now carry a real publisher URL.
recs = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/api/recommendations", timeout=20).read())
withurl = [r for r in recs if r["article"].get("url")]
print(f"recommendations: {len(recs)} — with a real publisher URL: {len(withurl)}")
if withurl: print("example:", withurl[0]["article"]["publisher"], "->", withurl[0]["article"]["url"])
print('-> reload the app -> Recommendations: each card\'s "Read article" opens the real publisher page')

In [ ]:
#@title (optional) 6 · Configure Google OAuth, then restart the web server
CLIENT_ID     = ""  #@param {type:"string"}
CLIENT_SECRET = ""  #@param {type:"string"}
import secrets, time, subprocess
assert PUBLIC_URL, "run step 4 first to get the public URL"
open("web/.env.local", "w").write(
    "RWE_BACKEND_URL=http://127.0.0.1:8000\n"
    f"GOOGLE_CLIENT_ID={CLIENT_ID}\n"
    f"GOOGLE_CLIENT_SECRET={CLIENT_SECRET}\n"
    f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}\n"
    f"NEXTAUTH_URL={PUBLIC_URL}\n"
    "RWE_DEV_LOGIN=1\n")   # keep the one-click demo login available alongside Google
# server-side env is read when the server starts, so a restart is enough (no rebuild).
web.terminate(); time.sleep(2)
web = subprocess.Popen(["npm", "start"], cwd="web",
                       stdout=open("web.log", "w"), stderr=subprocess.STDOUT)
print("Add this redirect URI to your Google OAuth client, then reload the app:")
print(" ", PUBLIC_URL + "/api/auth/callback/google")
print("web restarting:", "UP" if wait("http://127.0.0.1:3000/onboarding") else "check web.log")